# Data Mining from OWID Web Page

This notebook collects the **share of electricity generated by low-carbon sources** from the Our World in Data chart page and saves a clean country-level dataset for **2024**.

The output file is:
- `data/raw/owid_low_carbon_electricity_2024.csv`


## Imports


In [ ]:
import io
import re
from pathlib import Path

import pandas as pd
import requests


## Page and Output Paths


In [ ]:
page_url = "https://ourworldindata.org/grapher/share-electricity-low-carbon?country=OWID_WRL~CHN~IND~USA~BRA~RUS~ZAF&tab=chart"
csv_url = "https://ourworldindata.org/grapher/share-electricity-low-carbon.csv"
output_path = Path("data/raw/owid_low_carbon_electricity_2024.csv")

page_url, csv_url, output_path


## Download the OWID Page and Chart Data


In [ ]:
page_html = requests.get(page_url, timeout=30).text
chart_csv = requests.get(csv_url, timeout=30).text

print("Page length:", len(page_html))
print("CSV length:", len(chart_csv))


## Extract Page Metadata

We keep the OWID page metadata so the source is clearly documented in the assignment.


In [ ]:
last_updated_match = re.search(r"Last updated\s*</div>\s*<div[^>]*>\s*([A-Za-z]+ \d{1,2}, \d{4})", page_html)
date_range_match = re.search(r"Date range\s*</div>\s*<div[^>]*>\s*([0-9]{4}[–-][0-9]{4})", page_html)

last_updated = last_updated_match.group(1) if last_updated_match else None
date_range = date_range_match.group(1) if date_range_match else None

print("Last updated:", last_updated)
print("Date range:", date_range)


## Load, Filter, and Clean the Data

The OWID chart export contains many years and some regional aggregates. We keep only:
- `Year == 2024`
- rows with a valid country code


In [ ]:
df = pd.read_csv(io.StringIO(chart_csv))

owid_2024 = df[(df["Year"] == 2024) & (df["Code"].notna())].copy()
owid_2024 = owid_2024.rename(
    columns={
        "Entity": "country",
        "Code": "code",
        "Share of electricity from low-carbon sources": "low_carbon_electricity_share_pct"
    }
)

owid_2024["owid_last_updated"] = last_updated
owid_2024["owid_date_range"] = date_range
owid_2024["source_page_url"] = page_url
owid_2024["source_csv_url"] = csv_url

owid_2024 = owid_2024[[
    "country",
    "code",
    "Year",
    "low_carbon_electricity_share_pct",
    "owid_last_updated",
    "owid_date_range",
    "source_page_url",
    "source_csv_url"
]].rename(columns={"Year": "year"})

owid_2024 = owid_2024.sort_values("country").reset_index(drop=True)
owid_2024.head()


## Save the Final Dataset


In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
owid_2024.to_csv(output_path, index=False)

print(f"Saved {len(owid_2024)} rows to {output_path}")
owid_2024.sample(10, random_state=42)


## Summary

This notebook creates a clean scraped dataset that can be merged with the EV index as a **supplementary validation variable**.
